# Customer Segmentation Analysis

This notebook prepares customer data, segments customers with K-Means clustering, and compares the resulting groups.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Load the Data

In [ ]:
data = pd.read_csv(r"majorProject/marketing_campaign.csv", sep='\t')
data.columns

## 2. Clean and Engineer Features

In [ ]:
df = data.drop(['ID','Z_Revenue','Response'], axis=1)
df["TotalAcceptedCmp"] = (
    df["AcceptedCmp1"] +
    df["AcceptedCmp2"] +
    df["AcceptedCmp3"] + 
    df["AcceptedCmp4"] +
    df["AcceptedCmp5"]
)

In [ ]:
df.drop(['AcceptedCmp1','AcceptedCmp2','AcceptedCmp3','AcceptedCmp4','AcceptedCmp5'], axis=1, inplace=True)

In [ ]:
# df.info()
df

In [ ]:
df["Education"].unique()
df.drop_duplicates()
if pd.api.types.is_datetime64_any_dtype(df["Year_Birth"]):
    birth_year = df["Year_Birth"].dt.year
else:
    birth_year = pd.to_numeric(df["Year_Birth"], errors="coerce")
df["Age"] = 2026 - birth_year
df.drop(columns="Year_Birth", inplace=True)
df.head()

In [ ]:
spending_cols = [
    "MntWines",
    "MntFruits",
    "MntMeatProducts",
    "MntFishProducts",
    "MntSweetProducts",
    "MntGoldProds"
]

df["TotalSpending"] = df[spending_cols].sum(axis=1)
df.head()

In [ ]:
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="%d-%m-%Y")

In [ ]:
df["customerTenure"] = (
    df["Dt_Customer"].max() - df["Dt_Customer"]
)

df.drop(columns="Dt_Customer",inplace=True)
df.head()

In [ ]:
df.describe()["Age"]

In [ ]:
categorical_cols = df.select_dtypes(include=["object","string"]).columns
numerical_cols = df.select_dtypes(include="number").columns

In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    sparse_output=False,
    handle_unknown="ignore"
)
values = encoder.fit_transform(df[categorical_cols])
names = encoder.get_feature_names_out(categorical_cols)

df[names] = values
df.drop(columns=categorical_cols,inplace=True)
df.head()

In [ ]:
df["Income"] = df.Income.fillna(df["Income"].mean())

## 3. Prepare Data for Clustering

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Exclude timedelta columns because MinMaxScaler only accepts numeric values.
numerical_cols = df.select_dtypes(include="number", exclude="timedelta").columns
scaler = MinMaxScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()

In [ ]:
x = df[["Income","TotalSpending"]].fillna(df[["Income","TotalSpending"]].mean())

## 4. Build Customer Segments

In [ ]:
from sklearn.cluster import KMeans

wcss = []

for i in range(1,11):
    kmeans = KMeans(n_clusters=i,random_state=42)
    kmeans.fit(x)
    wcss.append(kmeans.inertia_)

plt.plot(range(1,11),wcss,marker="*")
plt.xlabel("Number Of Clusters")
plt.ylabel("WCSS")
plt.show()

In [ ]:
df["Income"] = df["Income"].fillna(df["Income"].mean())

In [ ]:
kmeans = KMeans(
    n_clusters=3,
    random_state=1,
    n_init=10
)
kmeans.fit(x)
df.loc[x.index, "Cluster"] = kmeans.labels_

## 5. Understand Customer Behaviour

In [ ]:
df[numerical_cols] = scaler.inverse_transform(df[numerical_cols])

cluster_summary = df.groupby("Cluster").agg(
    Customers=("Cluster", "size"),
    AverageAge=("Age", "mean"),
    AverageIncome=("Income", "mean"),
    AverageSpending=("TotalSpending", "mean")
).round(1)

cluster_summary

In [ ]:
cluster0 = df[df["Cluster"] == 0]
cluster1 = df[df["Cluster"] == 1]
cluster2 = df[df["Cluster"] == 2]

plt.figure(figsize=(7, 5))
plt.scatter(cluster0["Age"], cluster0["TotalSpending"], color="#2a9d8f", label="Cluster 0")
plt.scatter(cluster1["Age"], cluster1["TotalSpending"], color="#e9c46a", label="Cluster 1")
plt.scatter(cluster2["Age"], cluster2["TotalSpending"], color="#e76f51", label="Cluster 2")
plt.title("Age and spending")
plt.xlabel("Age")
plt.ylabel("Total spending")
plt.legend()
plt.show()

plt.figure(figsize=(7, 5))
plt.scatter(cluster0["Income"], cluster0["TotalSpending"], color="#2a9d8f", label="Cluster 0")
plt.scatter(cluster1["Income"], cluster1["TotalSpending"], color="#e9c46a", label="Cluster 1")
plt.scatter(cluster2["Income"], cluster2["TotalSpending"], color="#e76f51", label="Cluster 2")
plt.title("Income and spending")
plt.xlabel("Income")
plt.ylabel("Total spending")
plt.legend()
plt.show()

average_spending = df.groupby("Cluster")["TotalSpending"].mean()
average_spending.plot(kind="bar", color=["#2a9d8f", "#e9c46a", "#e76f51"])
plt.title("Average spending by cluster")
plt.xlabel("Customer cluster")
plt.ylabel("Average total spending")
plt.show()

## Conclusion

Customers with higher incomes generally spend more, while customers with lower incomes spend less. Cluster 1 contains the lowest-spending customers, Cluster 0 contains medium-spending customers, and Cluster 2 contains the highest-spending customers. Age is spread across all three groups, so it does not strongly explain spending behavior. The main difference between customers is their income and total spending.